In [2]:

import pandas as pd
import requests as r
import json as j
import ast as a
import pprint as pp



In [3]:

api_key = '3b4420f852a7b0ac65e8cec0f8d51ef4'

sport = 'baseball_mlb' # use the sport_key from the /sports endpoint below, or use 'upcoming' to see the next 8 games across all sports

regions = 'us' #multiple can be specified if comma delimited

markets = 'h2h,spreads'

bookmakers = 'us,us2'

odds_format = 'american'

date_format = 'iso'

all_format = 'true'
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#
# First get a list of in-season sports
#   The sport 'key' from the response can be used to get odds in the next request
#
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

sports_response = r.get(
    'https://api.the-odds-api.com/v4/sports', 
    params={
        'api_key': api_key,
        'regions': regions,
        'sport': sport,
        'all': all_format
    }
)


if sports_response.status_code != 200:
    print(f'Failed to get sports: status_code {sports_response.status_code}, response body {sports_response.text}')

else:
    print('List of in season sports:', sports_response.json())
    #dump the list of dictionaries provided by json output via api call in to a python data structure for further parsing
    sports_data = j.dumps(sports_response.json())
    #turn string to list
    sports_load_data = j.loads(sports_data)
    #iterate through list to get dictionaries
    df_sports = pd.DataFrame(sports_load_data)




    # Check the usage quota
    print('Remaining requests', sports_response.headers['x-requests-remaining'])
    print('Used requests', sports_response.headers['x-requests-used'])

df_sports.to_csv('list_of_sports.csv',index=True)


List of in season sports: [{'key': 'americanfootball_cfl', 'group': 'American Football', 'title': 'CFL', 'description': 'Canadian Football League', 'active': False, 'has_outrights': False}, {'key': 'americanfootball_ncaaf', 'group': 'American Football', 'title': 'NCAAF', 'description': 'US College Football', 'active': True, 'has_outrights': False}, {'key': 'americanfootball_ncaaf_championship_winner', 'group': 'American Football', 'title': 'NCAAF Championship Winner', 'description': 'US College Football Championship Winner', 'active': True, 'has_outrights': True}, {'key': 'americanfootball_nfl', 'group': 'American Football', 'title': 'NFL', 'description': 'US Football', 'active': False, 'has_outrights': False}, {'key': 'americanfootball_nfl_preseason', 'group': 'American Football', 'title': 'NFL Preseason', 'description': 'US Football', 'active': False, 'has_outrights': False}, {'key': 'americanfootball_nfl_super_bowl_winner', 'group': 'American Football', 'title': 'NFL Super Bowl Winn

In [4]:
odds_response = r.get(
    f'https://api.the-odds-api.com/v4/sports/{sport}/odds',
    params={
        'api_key': api_key,
        'regions': regions,
        'markets': markets,
        'oddsFormat': odds_format,
        'dateFormat': date_format,
    }
)

if odds_response.status_code != 200:
    print(f'Failed to get sports: status_code {odds_response.status_code}, response body {odds_response.text}')

else:
    #dump the list of dictionaries provided by json output via api call in to a python data structure for further parsing
    odds_data = odds_response.json()


    
    

    # Check the usage quota
    print('Remaining requests', odds_response.headers['x-requests-remaining'])
    print('Used requests', odds_response.headers['x-requests-used'])

Remaining requests 472
Used requests 28


In [5]:
rows_list = []
for i in odds_data:

    event_id = i['id']
    sport = i['sport_title']
    match_time = i['commence_time']
    home_team = i['home_team']
    away_team = i['away_team']

    for bookmakers in i['bookmakers']:

        bookmaker = bookmakers['title']
        bookmaker_last_update = bookmakers['last_update']

        for markets in bookmakers['markets']:
            
            market_event = markets['key']
            market_event_last_update = markets['last_update']

            for outcomes in markets['outcomes']:

                dictionary = {

                       'event_id': event_id,
                       'sport':sport,
                       'match_time':match_time,
                       'home_team':home_team,
                       'away_team':away_team,
                       'bookmaker':bookmaker,
                       'bookmaker_updated':bookmaker_last_update,
                       'market_event':market_event,
                       'market_event_update':market_event_last_update,
                       'outcome_name':outcomes['name'],
                       'price':outcomes['price'],
                       'points':outcomes.get('point')
                       

                      }
                
                rows_list.append(dictionary)

                
#df_full = pd.concat([pd.DataFrame([row]) for row in rows_list], ignore_index=True)
pd.DataFrame(rows_list).head(30)
        

#df_full.head()
    

bookmaker_actual = pd.DataFrame(rows_list)

bookmaker_actual.to_csv('bookmakers_actual.csv',index=True)


In [19]:
pivotted_odds = pd.pivot_table(bookmaker_actual,values = ['price','points'],index= ['event_id','market_event','outcome_name'],aggfunc=['mean','min','max']).reset_index()
pivotted_odds.columns = ['_'.join(col) for col in pivotted_odds.columns]
pivotted_odds.columns = [col[0:len(col)-1] if col[-1] == '_' else col for col in pivotted_odds.columns]
pivotted_odds.to_csv('pivotted_odds.csv',index=True)

pivotted_odds_and_bookmakers = pd.merge(bookmaker_actual,pivotted_odds,how='left',on=['event_id','market_event','outcome_name'])
pivotted_odds_and_bookmakers

,event_id,sport,match_time,home_team,away_team,bookmaker,bookmaker_updated,market_event,market_event_update,outcome_name,price,points,mean_points,mean_price,min_points,min_price,max_points,max_price
0,9aac4fc2c2231d9a8eab2a40c2a8cddf,MLB,2024-03-30T00:11:09Z,Houston Astros,New York Yankees,DraftKings,2024-03-30T03:03:35Z,h2h,2024-03-30T03:03:35Z,Houston Astros,3500,NaN,NaN,2940.000000,NaN,1900,NaN,4300
1,9aac4fc2c2231d9a8eab2a40c2a8cddf,MLB,2024-03-30T00:11:09Z,Houston Astros,New York Yankees,DraftKings,2024-03-30T03:03:35Z,h2h,2024-03-30T03:03:35Z,New York Yankees,-50000,NaN,NaN,-35600.000000,NaN,-100000,NaN,-8000
2,9aac4fc2c2231d9a8eab2a40c2a8cddf,MLB,2024-03-30T00:11:09Z,Houston Astros,New York Yankees,DraftKings,2024-03-30T03:03:35Z,spreads,2024-03-30T03:03:35Z,Houston Astros,154,4.5,4.944444,7.777778,4.5,-260,5.5,200
3,9aac4fc2c2231d9a8eab2a40c2a8cddf,MLB,2024-03-30T00:11:09Z,Houston Astros,New York Yankees,DraftKings,2024-03-30T03:03:35Z,spreads,2024-03-30T03:03:35Z,New York Yankees,-200,-4.5,-4.944444,-101.777778,-5.5,-260,-4.5,190
4,9aac4fc2c2231d9a8eab2a40c2a8cddf,MLB,2024-03-30T00:11:09Z,Houston Astros,New York Yankees,MyBookie.ag,2024-03-30T03:03:35Z,spreads,2024-03-30T03:03:35Z,Houston Astros,-225,5.5,4.944444,7.777778,4.5,-260,5.5,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
977,48f6a5704c93970245f3f0c01292cda2,MLB,2024-03-31T01:40:00Z,Seattle Mariners,Boston Red Sox,PointsBet (US),2024-03-30T03:03:35Z,spreads,2024-03-30T03:03:35Z,Seattle Mariners,140,-1.5,-1.458333,122.583333,-1.5,-104,-1.0,148
978,48f6a5704c93970245f3f0c01292cda2,MLB,2024-03-31T01:40:00Z,Seattle Mariners,Boston Red Sox,BetUS,2024-03-30T03:03:35Z,h2h,2024-03-30T03:03:35Z,Boston Red Sox,130,NaN,NaN,129.642857,NaN,120,NaN,133
979,48f6a5704c93970245f3f0c01292cda2,MLB,2024-03-31T01:40:00Z,Seattle Mariners,Boston Red Sox,BetUS,2024-03-30T03:03:35Z,h2h,2024-03-30T03:03:35Z,Seattle Mariners,-145,NaN,NaN,-149.642857,NaN,-157,NaN,-144
980,48f6a5704c93970245f3f0c01292cda2,MLB,2024-03-31T01:40:00Z,Seattle Mariners,Boston Red Sox,BetUS,2024-03-30T03:03:35Z,spreads,2024-03-30T03:03:35Z,Boston Red Sox,-165,1.5,1.458333,-165.833333,1.0,-180,1.5,-118


In [ ]:
col for col in col